# PillCounter AI — YOLO 알약 탐지 모델 학습 (Google Colab)

1. 메뉴 **런타임 > 런타임 유형 변경 > T4 GPU** 선택
2. 아래 셀을 위에서부터 순서대로 실행
3. 마지막 셀에서 받은 `pill_detector.tflite` (또는 `.onnx`) 를 프로젝트의 `app/src/main/assets/` 에 넣고 앱을 다시 빌드

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install "ultralytics>=8.3.0"

## 1) 파일 업로드
업로드할 것:
- `prepare_dataset.py`, `train.py` (프로젝트 `ml/` 폴더)
- 앱에서 내보낸 `pill_dataset_*.zip` (여러 개 가능)

AI Hub 데이터처럼 큰 데이터는 Google Drive 에 올린 뒤 아래 Drive 마운트 셀을 사용하세요.

In [ ]:
from google.colab import files
uploaded = files.upload()
print(list(uploaded))

In [ ]:
# (선택) Google Drive 마운트 - AI Hub 데이터를 Drive 에 올려둔 경우
from google.colab import drive
drive.mount('/content/drive')
AIHUB_DIR = '/content/drive/MyDrive/aihub_pills'   # 실제 경로로 수정 (없으면 빈 문자열)

## 2) 데이터셋 구성

In [ ]:
import glob, os
app_zips = ' '.join(glob.glob('/content/pill_dataset_*.zip'))
aihub = f'--aihub "{AIHUB_DIR}" --max-aihub 3000' if 'AIHUB_DIR' in globals() and AIHUB_DIR and os.path.isdir(AIHUB_DIR) else ''
app = f'--app {app_zips} --app-repeat 3' if app_zips else ''
!python prepare_dataset.py {app} {aihub} --out /content/datasets/pills

## 3) 학습 + 평가 + TFLite 변환
- `--model yolo11n.pt` : 가장 가볍고 빠름 (스마트폰 실시간용, 권장)
- `--model yolo11s.pt` : 더 정확하지만 약 3배 느림
- 데이터 수백 장 기준 T4 GPU 로 약 20~60분

In [ ]:
!python train.py --data /content/datasets/pills/data.yaml --model yolo11n.pt --epochs 120 --imgsz 640 --device 0

## 4) 결과 확인 및 모델 다운로드

In [ ]:
from IPython.display import Image, display
for f in ['results.png', 'val_batch0_pred.jpg']:
    p = f'/content/runs/detect/pill/{f}'
    if os.path.exists(p): display(Image(p, width=900))

In [ ]:
import shutil
w = '/content/runs/detect/pill/weights'
cands = sorted(glob.glob(f'{w}/**/*float32.tflite', recursive=True)) or sorted(glob.glob(f'{w}/**/*.tflite', recursive=True))
if cands:
    shutil.copy(cands[0], '/content/pill_detector.tflite')
    files.download('/content/pill_detector.tflite')
else:  # TFLite 변환 실패 시 ONNX 로 대체 (앱이 둘 다 지원)
    !python train.py --export-only {w}/best.pt --format onnx
    shutil.copy(f'{w}/best.onnx', '/content/pill_detector.onnx')
    files.download('/content/pill_detector.onnx')
files.download(f'{w}/best.pt')   # 추가 학습용 원본 가중치
